# BanglaSQL — auxiliary table-classification head

Trains BanglaT5 twice on identical data and compares:

| run | what it is |
|---|---|
| **baseline** | plain BanglaT5 fine-tuning (λ = 0) |
| **head** | + auxiliary head predicting the tables a question needs (λ = 0.5) |

```
question ─► T5 encoder ─► mean-pool ─► Linear(768→6) ─► tables
                  └──► T5 decoder ─► SQL
```

Loss: `L = L_seq2seq + 0.5 × BCE(tables)`. At inference the predicted tables pick between beams,
and a **column check** skips beams returning columns the question never mentions.

Section 6 reports every step in one table: baseline → + column check → + table head → + both.

> Runtime → Change runtime type → **T4 GPU**. Two trainings take about **1.5–2.5 hours** in total.
> Each checkpoint is saved to Drive as soon as it finishes, so a disconnect loses at most one run.

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to a T4 runtime')

In [ ]:
REPO_URL = 'https://github.com/mustafiz-07/BanglaSQL.git'

!rm -rf banglasql
!git clone -q -b model_contribution {REPO_URL} banglasql
%cd banglasql/aux_head

import os
if not os.path.exists('table_head.py'):
    raise SystemExit('aux_head/ is not on the remote branch yet — commit and push it, then re-run this cell.')
!pip install -q -r requirements_colab.txt
print('Ready.')

In [ ]:
# Everything this notebook produces is kept here, so a disconnect loses nothing already finished.
DRIVE = '/content/drive/MyDrive/BanglaSQL/aux_head'
BASE_CKPT = DRIVE + '/baseline'
HEAD_CKPT = DRIVE + '/head'
os.makedirs(DRIVE, exist_ok=True)

def trained(path):
    return os.path.exists(os.path.join(path, 'model.safetensors'))

print('baseline on Drive:', trained(BASE_CKPT))
print('head on Drive    :', trained(HEAD_CKPT))

## 2 — Database and dataset
Deterministic: the check below confirms Colab rebuilt exactly the data this was developed on.

In [ ]:
!python create_database.py | tail -3
!python build_dataset.py | grep -E 'OK|Train :|Dev   :|Test  :'
!python preprocess_check.py | grep -E 'Recommended|Config saved'

import hashlib
EXPECTED = '8c15653fa1aac7a0e7a544f9cb24f9d3'
actual = hashlib.md5(open('data/dataset_test.json', 'rb').read()).hexdigest()
print()
print('test set md5:', actual, '-> MATCHES' if actual == EXPECTED else '-> DIFFERENT data, results not comparable')

## 3 — Train the baseline (λ = 0)

The same code with the head switched off, so the two runs differ in exactly one thing.
Skipped automatically if a baseline is already on Drive.

In [ ]:
import shutil

def keep(label, dest):
    shutil.copytree('checkpoints/best_model', dest, dirs_exist_ok=True)
    shutil.copy('logs/train_history.json', os.path.join(dest, 'train_history.json'))
    print(label, 'saved to', dest)

if trained(BASE_CKPT):
    print('Baseline already on Drive — skipping training.')
else:
    !rm -rf checkpoints logs
    !BANGLASQL_AUX_WEIGHT=0 python train.py
    keep('baseline', BASE_CKPT)

## 4 — Train with the table head (λ = 0.5)

Watch for `aux_loss` in the loss lines — it starts near 0.69 (an untrained binary classifier) and
should fall as the head learns.

In [ ]:
if trained(HEAD_CKPT):
    print('Head model already on Drive — skipping training.')
else:
    !rm -rf checkpoints logs
    !BANGLASQL_AUX_WEIGHT=0.5 python train.py
    keep('head', HEAD_CKPT)

print('table_head.pt present:', os.path.exists(os.path.join(HEAD_CKPT, 'table_head.pt')))

### Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

def history(path):
    return json.load(open(os.path.join(path, 'train_history.json')))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))
for label, path in [('baseline', BASE_CKPT), ('head', HEAD_CKPT)]:
    h = history(path)
    acc = [(x['epoch'], x['eval_execution_accuracy']) for x in h if 'eval_execution_accuracy' in x]
    loss = [(x['epoch'], x['loss']) for x in h if 'loss' in x]
    ax1.plot(*zip(*loss), label=label)
    ax2.plot(*zip(*acc), label=label)
aux = [(x['epoch'], x['aux_loss']) for x in history(HEAD_CKPT) if 'aux_loss' in x]
if aux:
    ax3.plot(*zip(*aux), color='purple')
ax1.set_title('training loss'); ax2.set_title('dev execution accuracy'); ax3.set_title('auxiliary table loss (head run)')
for ax in (ax1, ax2, ax3):
    ax.set_xlabel('epoch')
ax1.legend(); ax2.legend()
plt.tight_layout(); plt.show()

## 5 — Evaluate both on the test set

Each model is scored under several selection rules, all choosing from the **same** beams, so the
difference between two rows is that rule alone:

| rule | chooses |
|---|---|
| exec-guided | the highest-ranked beam that runs |
| + column check | …skipping beams that return a column the question never mentions |
| + table head | …preferring the beam whose tables match the head's prediction |

The table head fixes *which tables* a query reads; the column check fixes *which columns* it
returns. Neither requires retraining — only the head's own training does.

In [ ]:
!python evaluate.py --split test --model "{BASE_CKPT}" --tag baseline | sed -n '/RESULTS/,$p'

In [ ]:
!python evaluate.py --split test --model "{HEAD_CKPT}" --tag head | sed -n '/RESULTS/,$p'

## 6 — Comparison

In [ ]:
import pandas as pd

base = json.load(open('logs/test_baseline_results.json', encoding='utf-8'))
head = json.load(open('logs/test_head_results.json', encoding='utf-8'))

# (label, results file, selection rule) — each row adds one thing to the row above it.
LADDER = [
    ('baseline',                          base, 'exec_guided'),
    ('baseline + column check',           base, 'column_check'),
    ('head model (training only)',        head, 'exec_guided'),
    ('head + table rerank',               head, 'table_head'),
    ('head + table rerank + column check', head, 'table_head+column_check'),
]

rows = {}
for label, res, arm in LADDER:
    m, fc = res['arms'][arm]['metrics'], res['arms'][arm]['failure_categories']
    rows[label] = {
        'exec accuracy': f"{100 * m['execution_accuracy']:.1f}%",
        'exact match': f"{100 * m['exact_match']:.1f}%",
        'validity': f"{100 * m['validity_rate']:.1f}%",
        'wrong_table': fc.get('wrong_table', 0),
        'wrong_select_columns': fc.get('wrong_select_columns', 0),
    }
display(pd.DataFrame(rows).T)

In [ ]:
# What each step fixed and broke, relative to the step before it, on the same model.
for model_label, res in [('baseline', base), ('head', head)]:
    for step, r in res['steps'].items():
        print(f"{model_label:<9} {step:<44} {r['fixed']:3d} fixed [{r['fixed_templates']} templates]"
              f"   {r['broke']:3d} broke [{r['broke_templates']} templates]")

In [ ]:
# All failure categories: plain baseline vs the full system.
b_fc = base['arms']['exec_guided']['failure_categories']
h_fc = head['arms']['table_head+column_check']['failure_categories']
cats = sorted(set(b_fc) | set(h_fc), key=lambda c: -b_fc.get(c, 0))
display(pd.DataFrame({'baseline': [b_fc.get(c, 0) for c in cats],
                      'full system': [h_fc.get(c, 0) for c in cats]}, index=cats))

In [ ]:
# How well the head itself predicts tables.
h = head['table_head']
print('table-set exact accuracy:', f"{100 * h['table_set_accuracy']:.1f}%")
display(pd.DataFrame(h['per_table']).T)

### Per-example: plain baseline → full system

Augmented variants of one question are near-duplicates, so fixes are also counted per **base template**
— twelve fixes from one template are one piece of evidence, not twelve.

In [ ]:
from collections import Counter

pairs = json.load(open('data/dataset_test.json', encoding='utf-8'))
bp = json.load(open('logs/test_baseline_predictions.json', encoding='utf-8'))
hp = json.load(open('logs/test_head_predictions.json', encoding='utf-8'))

def outcome(record, arm):
    return record['arms'][arm]

fixed, broke, shown = Counter(), Counter(), []
for p, b, x in zip(pairs, bp, hp):
    before = outcome(b, 'exec_guided')
    after = outcome(x, 'table_head+column_check')
    if after['correct'] and not before['correct']:
        fixed[p['base_template_id']] += 1
        shown.append((p, before, after, x))
    elif before['correct'] and not after['correct']:
        broke[p['base_template_id']] += 1

print(f'fixed: {sum(fixed.values())} examples from {len(fixed)} base templates  {dict(fixed)}')
print(f'broke: {sum(broke.values())} examples from {len(broke)} base templates  {dict(broke)}')
print()
for p, before, after, x in shown[:5]:
    print('Q         ', p['bangla_question'])
    print('baseline  ', before['sql'][:110])
    print('full      ', after['sql'][:110], '  tables:', x.get('predicted_tables'))
    print()

> **Reading this:** the test set has 347 examples but only 36 base templates, so an execution-accuracy
> difference under ~10 points is within noise. The stronger evidence is the `wrong_table` count, the
> head's own table accuracy, and fixed/broke counted per base template.

## 7 — Save results to Drive

In [ ]:
shutil.copytree('logs', DRIVE + '/logs', dirs_exist_ok=True)
print('Saved to', DRIVE + '/logs')

## 8 — Try it
Generates 4 beams, shows the tables the head predicts, and the query the full system chose
(table head + column check).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from common import load_config, open_readonly_db, pick_executable, run_sql, tables_in
from evaluate import generate_candidates
from table_head import load_head, predict_tables

tok = AutoTokenizer.from_pretrained(HEAD_CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(HEAD_CKPT)
if torch.cuda.is_available():
    model = model.cuda()
config = load_config(HEAD_CKPT)
tbl_head = load_head(HEAD_CKPT)
con = open_readonly_db()

def ask(question):
    beams = generate_candidates(model, tok, [question], config, 4)[0]
    predicted = predict_tables(model, tbl_head, tok, [question], config)[0]
    plain = pick_executable(beams, con)
    chosen = pick_executable(beams, con, predicted, question=question)
    print('Q               ', question)
    print('predicted tables', sorted(predicted))
    for i, b in enumerate(beams, 1):
        mark = ' <- chosen' if b == chosen else ''
        print(f'  beam {i}  {sorted(tables_in(b))!s:<32} {b[:80]}{mark}')
    if plain != chosen:
        print('  (plain exec-guided would have chosen beam', beams.index(plain) + 1, ')')
    # run_sql returns the error instead of raising, so a question with no valid beam
    # reports that rather than crashing the demo.
    rows, err = run_sql(con, chosen)
    if err:
        print('  -> did not execute:', err)
    else:
        print(f'  -> {len(rows)} rows', rows[:5])

ask('প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে?')

In [ ]:
ask("'Fall 2023' সেমিস্টারে অনুষ্ঠিত কোর্সের নাম দেখাও।")